# Declarative geoprocessing pipelines (`pyramids.processing`)

This notebook teaches the **`pyramids.processing`** layer — a Whitebox/QGIS-Processing-style way to describe a
reusable, serializable **pipeline** of pyramids operations and run it (batched) over one or many inputs.

By the end you will be able to:

- discover the available **tools** and read their parameter schemas;
- build a **`Pipeline`** — an ordered chain of `(tool, params)` steps — and run it with **`run`**;
- chain across object types (a `FeatureCollection` step whose output feeds a `Dataset` step);
- save a pipeline to a portable **YAML** file and load it back;
- run a pipeline **batched** over a folder of rasters with an error policy;
- read the **provenance** of a run.

API reference: `pyramids.processing.Pipeline`, `pyramids.processing.run`,
`pyramids.processing.get_registry` / `tool_names`.

## Setup

We import the pipeline surface plus `Dataset` / `FeatureCollection`, set a single notebook-relative path to the
example data, and enable inline plotting. Everything else in the notebook is about the feature, not the plumbing.

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd

from pyramids.dataset import Dataset
from pyramids.feature import FeatureCollection
from pyramids.processing import Pipeline, get_registry, run

DATA = Path("../../../examples/data")

## The tool registry

A pipeline can only reference **registered tools**: existing pyramids ops made addressable by name, each with a
parameter schema and a *receiver type* (does it run on a `Dataset` or a `FeatureCollection`?) and a *return type*.
`get_registry()` returns that catalogue — here it is as a table.

In [ ]:
registry_table = pd.DataFrame(
    [
        {
            "tool": name,
            "receiver": spec.receiver,
            "returns": spec.returns,
            "params": ", ".join(p.name for p in spec.params) or "-",
            "description": spec.description,
        }
        for name, spec in sorted(get_registry().items())
    ]
)
registry_table

Two things to notice: tools are split by **receiver** (raster `Dataset` ops vs vector `FeatureCollection`
ops), and a few return **`Array`** — those are the terrain ops (`slope`/`aspect`/`hillshade`/`focal_*`) whose
numpy output the runner re-wraps into a georeferenced single-band `Dataset`, so they stay writable and chainable.

## Quickstart — a cross-receiver pipeline

The headline capability: a pipeline can change object type mid-chain. Here `interpolate_to_raster` runs on a
**`FeatureCollection`** of points and returns a **`Dataset`**, and the next step, `slope`, runs on that raster —
the runner dispatches each step to the right object automatically.

We use the Coello rain-gauge points. This sample carries no measured attribute, so we grid the northing (`y`)
purely to get a continuous surface to work with; in practice you would grid rainfall, elevation, etc.

In [ ]:
gauges = FeatureCollection.read_file(str(DATA / "coello-gauges.geojson"))
gauges.plot(column="y", markersize=80, legend=True)

We define the pipeline as a plain list of `(tool, params)` steps. It is validated **at construction** — an
unknown tool or a bad parameter fails right here, not halfway through a long run.

In [ ]:
quickstart = Pipeline(
    [
        ("interpolate_to_raster", {"column": "y", "cell_size": 2000.0}),
        ("slope", {}),
    ]
)
quickstart

`run` executes the pipeline over the input and returns a `RunResult`. Because the final step is a terrain op,
its array is materialized back into a georeferenced `Dataset` — so the output is a real raster we can plot.

In [ ]:
result = run(quickstart, gauges)
slope_raster = result.outputs[0]
slope_raster.plot(title="Slope of the interpolated gauge surface")

The gauge points became a continuous IDW surface, and `slope` turned that into a slope raster — one linear
chain, two receiver types, no manual glue. `result.outputs` holds the final object per input; `result.failures`
would hold `(input, error)` pairs under the default skip policy (none here).

## A raster pipeline on a real DEM

Now a `Dataset`-only chain on a real elevation model (the Rhine 5 km DEM). First, load and look at the DEM.

In [ ]:
dem = Dataset.read_file(str(DATA / "dem" / "DEM5km_Rhine_burned_fill.tif"))
dem.plot(title="Rhine DEM (filled)")

A two-step terrain pipeline: smooth the DEM with a 3x3 mean (`focal_mean`), then compute `hillshade`. Each
step is one idea; the chain reads top to bottom.

In [ ]:
terrain = Pipeline(
    [
        ("focal_mean", {"radius": 1}),
        ("hillshade", {"azimuth": 315.0, "altitude": 45.0}),
    ]
)
hillshaded = run(terrain, dem).outputs[0]
hillshaded.plot(title="Hillshade of the smoothed DEM", cmap="gray")

The result is a shaded-relief raster. Swapping or reordering steps is a one-line edit to the pipeline list —
that is the point of describing the workflow as *data* rather than hand-written calls.

## Serialize a pipeline (the portable "model")

`to_yaml` writes the pipeline to a small, version-controllable file; `from_yaml` loads it back. Only
serialization-safe (scalar) parameters are allowed — `to_yaml` refuses to write anything it could not load back.

In [ ]:
yaml_path = Path("terrain.yaml")
terrain.to_yaml(str(yaml_path))
print(yaml_path.read_text())

Loading it reconstructs an equivalent pipeline (validated again against the current registry).

In [ ]:
reloaded = Pipeline.from_yaml(str(yaml_path))
reloaded == terrain

`terrain.yaml` was just a scratch file for the demo — remove it so running this notebook leaves the tree
clean.

In [ ]:
yaml_path.unlink(missing_ok=True)

## Batch over many inputs

`run` accepts a single object, a `DatasetCollection`, or a **glob** of files. Here we smooth every raster in a
folder in one call. The `on_error` policy is `"skip"` by default — failures are collected in `result.failures`
and the batch keeps going; use `"raise"` to fail fast.

In [ ]:
smooth = Pipeline([("focal_mean", {"radius": 2})])
batch = run(smooth, str(DATA / "crop_aligned_folder" / "*.tif"))
len(batch.outputs), len(batch.failures)

Every input produced one output raster. Here is the first one.

In [ ]:
batch.outputs[0].plot(title="Smoothed (focal_mean, radius=2) — first tile")

## Vector tools

The vector receiver has its own tools. `voronoi` builds Thiessen polygons around the gauge points; `quadtree`
adaptively bins them. Both take a `FeatureCollection` and return one.

In [ ]:
cells = run(Pipeline([("voronoi", {})]), gauges).outputs[0]
cells.plot(edgecolor="black", facecolor="none")

Each polygon is the region closest to one gauge — a common first step for turning point observations into
areal coverage.

## Provenance

Every run records, per step, the tool, its params, and the wall-clock time — and can re-emit the exact pipeline
that produced an output (reproducibility).

In [ ]:
prov = result.provenance[0]
pd.DataFrame(
    [
        {"tool": s.tool, "params": s.params, "seconds": round(s.seconds, 4)}
        for s in prov.steps
    ]
)

`prov.to_pipeline()` returns a `Pipeline` equal to the one we ran, so a result's recipe can be replayed or
saved:

In [ ]:
prov.to_pipeline() == quickstart

## The command line

The same three pieces are on the CLI:

| command | does |
|---------|------|
| `pyramids tools` | list the registered tools |
| `pyramids tool <name>` | print one tool's parameter schema |
| `pyramids run pipeline.yaml --inputs "*.tif" --out dir/` | run a pipeline (batched) over inputs, writing to `dir/` |

So a pipeline authored and saved in Python (or a GUI that reads the same YAML) runs unchanged from a shell script.

## Takeaway

You built and ran declarative pipelines over real data:

- **tools** are named, self-describing ops (`get_registry` / `tool_names`);
- a **`Pipeline`** is a validated `(tool, params)` chain that can cross receiver types (points → raster → slope);
- **`run`** executes it over one input, a collection, or a glob, with a skip/raise error policy;
- pipelines **serialize** to YAML and back, and every run carries **provenance** you can replay.

Next: register your own `ToolSpec` to add a tool, or drive `run(..., out=..., parallel=True)` to process a large
folder across a process pool.